<a href="https://colab.research.google.com/github/01PrathamS/Transformers_from_scratch/blob/main/notebooks/LayerNormaliation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
from torch import nn

In [2]:
inputs = torch.Tensor([[[0.2, 0.1, 0.3], [0.5, 0.1, 0.1]]])
B, S, E = inputs.size()
inputs = inputs.permute(1, 0, 2)   # inputs = inputs.reshape(S, B, E)
inputs.size()

torch.Size([2, 1, 3])

In [3]:
parameter_shape = inputs.size()[-2:]
gamma = nn.Parameter(torch.ones(parameter_shape))
beta = nn.Parameter(torch.zeros(parameter_shape))

In [4]:
gamma.size(), beta.size()

(torch.Size([1, 3]), torch.Size([1, 3]))

In [5]:
dims = [-(i + 1) for i in range(len(parameter_shape))]
dims

[-1, -2]

In [6]:
mean = inputs.mean(dim=dims, keepdim=True)
mean.size(), mean

(torch.Size([2, 1, 1]),
 tensor([[[0.2000]],
 
         [[0.2333]]]))

In [7]:
var = ((inputs - mean) ** 2).mean(dim=dims, keepdim=True)
epsilon = 1e-5
std = (var + epsilon).sqrt()
std

tensor([[[0.0817]],

        [[0.1886]]])

In [8]:
y = (inputs - mean) / std
y

tensor([[[ 0.0000, -1.2238,  1.2238]],

        [[ 1.4140, -0.7070, -0.7070]]])

In [9]:
out = gamma * y + beta
out

tensor([[[ 0.0000, -1.2238,  1.2238]],

        [[ 1.4140, -0.7070, -0.7070]]], grad_fn=<AddBackward0>)

In [10]:
class LayerNormalization():
    def __init__(self, parameters_shape, eps=1e-5):
        self.parameters_shape=parameters_shape
        self.eps=eps
        self.gamma = nn.Parameter(torch.ones(parameters_shape))
        self.beta =  nn.Parameter(torch.zeros(parameters_shape))

    def forward(self, input):
        dims = [-(i + 1) for i in range(len(self.parameters_shape))]
        mean = inputs.mean(dim=dims, keepdim=True)
        print(f"Mean \n ({mean.size()}): \n {mean}")
        var = ((inputs - mean) ** 2).mean(dim=dims, keepdim=True)
        std = (var + self.eps).sqrt()
        print(f"Standard Deviation \n ({std.size()}): \n {std}")
        y = (inputs - mean) / std
        print(f"y \n ({y.size()}) = \n {y}")
        out = self.gamma * y  + self.beta
        print(f"out \n ({out.size()}) = \n {out}")
        return out

In [11]:
batch_size = 3
sentence_length = 5
embedding_dim = 8
inputs = torch.randn(sentence_length, batch_size, embedding_dim)

print(f"input \n ({inputs.size()}) = \n {inputs}")

input 
 (torch.Size([5, 3, 8])) = 
 tensor([[[ 0.0587,  1.7285, -0.7499, -1.0340,  0.2747,  1.0284, -0.9253,
          -0.3402],
         [-0.0870, -0.3452, -0.1522,  2.5325, -0.3359,  0.1787, -0.1145,
           0.3975],
         [ 0.8281, -0.7083,  0.1527, -0.4818, -0.6148, -0.7900, -1.3922,
           1.0100]],

        [[ 0.1185, -0.3092,  0.5536, -1.6824,  0.1428, -1.0059,  0.3042,
          -1.0651],
         [-1.0600, -0.0791,  0.0122,  0.8671, -0.1391, -2.3366, -0.6332,
          -0.8576],
         [-1.0022, -0.9000,  0.7298, -2.9414, -0.4966, -0.6949,  1.2805,
           1.6006]],

        [[ 0.1495, -0.8118,  0.3051, -0.6966,  2.5329, -0.8135, -2.0844,
          -0.7645],
         [ 0.0774,  0.4979, -0.4669, -2.6591,  0.7863, -0.6830,  0.5105,
           0.3514],
         [-0.5551, -0.3315, -0.1646, -0.2716,  0.4571,  0.6814,  0.4454,
          -1.0588]],

        [[-0.2354,  0.9255, -1.3020, -0.9071, -0.9391,  1.5778, -0.8790,
          -1.1257],
         [ 0.3638, -0.1114, 

In [12]:
layer_norm = LayerNormalization(inputs.size()[-1:])

In [13]:
out = layer_norm.forward(inputs)

Mean 
 (torch.Size([5, 3, 1])): 
 tensor([[[ 0.0051],
         [ 0.2593],
         [-0.2496]],

        [[-0.3679],
         [-0.5283],
         [-0.3030]],

        [[-0.2729],
         [-0.1982],
         [-0.0997]],

        [[-0.3606],
         [ 0.0992],
         [ 0.1179]],

        [[ 0.4677],
         [-0.1497],
         [ 0.2153]]])
Standard Deviation 
 (torch.Size([5, 3, 1])): 
 tensor([[[0.9170],
         [0.8904],
         [0.7830]],

        [[0.7435],
         [0.8849],
         [1.3782]],

        [[1.2585],
         [1.0427],
         [0.5514]],

        [[0.9875],
         [0.4403],
         [1.0667]],

        [[0.8748],
         [0.8963],
         [1.1803]]])
y 
 (torch.Size([5, 3, 8])) = 
 tensor([[[ 0.0585,  1.8793, -0.8233, -1.1331,  0.2940,  1.1159, -1.0146,
          -0.3766],
         [-0.3889, -0.6789, -0.4621,  2.5532, -0.6685, -0.0904, -0.4197,
           0.1553],
         [ 1.3762, -0.5858,  0.5137, -0.2966, -0.4665, -0.6902, -1.4593,
           1.6085]],



In [14]:
out[0].mean(), out[0].std()

(tensor(-1.4901e-08, grad_fn=<MeanBackward0>),
 tensor(1.0215, grad_fn=<StdBackward0>))